In [ ]:
from matplotlib import pyplot as plt
from scipy.stats import norm
import numpy as np 
import csv
import math
import sys
import VSPFunctions as vsp
import pandas as pd
import seaborn as sns
from scipy.optimize import curve_fit
from scipy.stats import lognorm
import os
from astropy.table import Table
from astropy.io import fits

directory = "/users/cdcook/VSP/graphics/"

sns.set_theme(style="darkgrid")

In [ ]:
# Function to check if a specific flag is true
def flag_is_true(row, flag):
    return row.get(flag, False)

def KronCut(kronBand, kronDist, dataDF):
    if (kronBand == 'g'):
        kronName = 'gMeanKronMag'
        psfName = 'gMeanPSFMag'
        kronCutName = 'gKron'
    if (kronBand == 'r'):
        kronName = 'rMeanKronMag'
        psfName = 'rMeanPSFMag'
        kronCutName = 'rKron'
    if (kronBand == 'i'):
        kronName = 'iMeanKronMag'
        psfName = 'iMeanPSFMag'
        kronCutName = 'iKron'
    if (kronBand == 'z'):
        kronName = 'zMeanKronMag'
        psfName = 'zMeanPSFMag'
        kronCutName = 'zKron'
    if (kronBand == 'y'):
        kronName = 'yMeanKronMag'
        psfName = 'yMeanPSFMag'
        kronCutName = 'yKron'
        
    # Condition 1: psfName - kronName should be less than kronDist
    condition1 = abs(dataDF[psfName] - dataDF[kronName]) < kronDist
    # Apply both conditions to the DataFrame
    dataDF = dataDF[condition1]
    
    return dataDF, kronName, psfName, kronCutName

def BitFlagCut(bitFlags, dataDF):
    dataDF = dataDF[dataDF['Flags'] == bitFlags]
    return dataDF

def ColorCut(dataDF, sub1, sub2):
    if (sub1 == 'gr'):
        band1a = 'gMeanPSFMag'
        band1b = 'rMeanPSFMag'
        band1name = "g-r"
        
    elif (sub1 == 'gi'):
        band1a = 'gMeanPSFMag'
        band1b = 'iMeanPSFMag'
        band1name = "g-i"
        
    elif (sub1 == 'ri'):
        band1a = 'rMeanPSFMag'
        band1b = 'iMeanPSFMag'
        band1name = "r-i"
        
    if (sub2 == 'gr'):
        band2a = 'gMeanPSFMag'
        band2b = 'rMeanPSFMag'
        band2name = "g-r"
    
    elif (sub2 == 'gi'):
        band2a = 'gMeanPSFMag'
        band2b = 'iMeanPSFMag'
        band2name = "g-i"
        
    elif (sub2 == 'ri'):
        band2a = 'rMeanPSFMag'
        band2b = 'iMeanPSFMag'
        band2name = "r-i"
        
    x = []
    y = []
    for index, row in dataDF.iterrows():
        x.append(row[band1a] - row[band1b])
        y.append(row[band2a] - row[band2b])

    dataDF.insert(0, 'color1', x)
    dataDF.insert(1, 'color2', y)
    return dataDF, band1a, band1b, band2a, band2b, band1name, band2name

In [ ]:
def find_limiting_magnitude(data):
    if 'M' in data.columns.names and 'JD' in data.columns.names:
        magnitudes = data['M']
        julian_dates = data['JD']
        num_exposures = magnitudes.shape[2]  # Number of exposures

        # Debugging statement to check the shape of the Julian Dates array
        print("Shape of Julian Dates array:", julian_dates.shape)

        results = []
        for exposure_index in range(num_exposures):
            # Get magnitudes for the current exposure
            magnitudes_exposure = magnitudes[0, :, exposure_index]

            # Get the Julian Date for the current exposure
            julian_date = julian_dates[0, exposure_index] if julian_dates.ndim > 1 else julian_dates[exposure_index]

            # Apply condition to ignore magnitudes < 5 or > 25
            valid_magnitudes = magnitudes_exposure[(magnitudes_exposure >= 5) & (magnitudes_exposure <= 25)]

            # Find the maximum magnitude within the valid range
            if len(valid_magnitudes) > 0:
                limiting_mag = np.max(valid_magnitudes)
            else:
                limiting_mag = np.nan  # Handle case where all valid magnitudes are filtered out

            results.append((julian_date, limiting_mag))

        # Remove the last item from the list
        results.pop()

        julian_dates, limiting_mags = zip(*results)
        
        return julian_dates, limiting_mags
    else:
        print("M or JD column not found in the data.")
        return [], []

In [ ]:
field = 'sky0001_1d'
dates = ['0824', '0825', '0901', '0902', '0903', '0905', '0906', '0928', '0929', '1001', '1002', '1003', '1004', '1005', '1006']

In [ ]:
field = 'xtetrans_1a'
dates = ['0409', '0410', '0413', '0414', '0415', '0416', '0417']

In [ ]:
for n in dates: 
    name = str(n) + 'dataDF'
    slopeList = []
    ABoffsetList = []
    countsList = []
    exposures = []    
    kronCutTF = True
    
    file_path = f'/lustre/work/client/users/cdcook/fits_structs/00{n}_{field}_match.fit'
    
    with fits.open(file_path) as hdul:
        if len(hdul) > 1:
            data_hdu = hdul[1]
            data = data_hdu.data
            julian_dates, limiting_mags = find_limiting_magnitude(data)
            for i, (jd, mag) in enumerate(zip(julian_dates, limiting_mags)):
                print(f"Exposure {i+1}: Julian Date = {jd}, Limiting Magnitude = {mag:.2f}")
        else:
            print("No data found in the first extension HDU.")

    for j in range(len(julian_dates)):
        exposure = j
        title = f'/lustre/work/client/users/cdcook/VSPData/meanFields/meanSky0001_1d{n}Files/Pan00{n}_1d_exp{str(exposure)}.csv'
        panDF = pd.read_csv(title)
        panDF = panDF.drop_duplicates(subset=['RA', 'Dec'])
        panDF = panDF[(panDF['Mag'] >= 5) & (panDF['Mag'] <= 25)]
        panDF = panDF[(panDF['gMeanPSFMag'] != -999) & (panDF['rMeanPSFMag'] != -999) & (panDF['iMeanPSFMag'] != -999) & 
                      (panDF['zMeanPSFMag'] != -999) & (panDF['yMeanPSFMag'] != -999) & (panDF['gMeanKronMag'] != -999) & 
                      (panDF['rMeanKronMag'] != -999) & (panDF['iMeanKronMag'] != -999) & (panDF['zMeanKronMag'] != -999) & 
                      (panDF['yMeanKronMag'] != -999)]
        
        def calculate_flux(magnitude):
            return np.power(10, (magnitude + 48.6) / -2.5)
        
        panDF['gflux'] = panDF['gMeanPSFMag'].apply(calculate_flux)
        panDF['rflux'] = panDF['rMeanPSFMag'].apply(calculate_flux)
        panDF['iflux'] = panDF['iMeanPSFMag'].apply(calculate_flux)
        panDF['zflux'] = panDF['zMeanPSFMag'].apply(calculate_flux)
        panDF['yflux'] = panDF['yMeanPSFMag'].apply(calculate_flux)
        
        panDF['totalFlux'] = (
            panDF['gflux'] * 0.1212 + 
            panDF['rflux'] * 0.1463 + 
            panDF['iflux'] * 0.1435 + 
            panDF['zflux'] * 0.098 + 
            panDF['yflux'] * 0.0393
        ) / 0.5483
        
        panDF['logpart'] = np.log10(panDF['totalFlux'] / 3631e-23)
        panDF['pseudoBoloMag'] = -2.5 * panDF['logpart']
        
        flux_columns = ['gflux', 'rflux', 'iflux', 'zflux', 'yflux']
        panDF.drop(columns=flux_columns, inplace=True)
        
        panDF['Difference'] = panDF['pseudoBoloMag'] - panDF['Mag']
        panDF['objInfoFlag'] = panDF['objInfoFlag'].apply(lambda x: f'0x{x:08X}')
        panDF['qualityFlag'] = panDF['qualityFlag'].apply(lambda x: f'0x{x:08X}')
    
        panDF, band1a, band1b, band2a, band2b, band1name, band2name = ColorCut(panDF, 'gr', 'ri')
        if kronCutTF:
            panDF, kronName, psfName, kronCutName = KronCut(kronBand='g', kronDist=0.5, dataDF=panDF)
        
        panDF2 = panDF
        params = np.polyfit(panDF2['Mag'], panDF2['pseudoBoloMag'], 1, full=False, cov=True)
    
        slopeList.append(params[0][0])
        ABoffsetList.append(params[0][1])
        countsList.append(len(panDF2))
        exposures.append(j)
    
    data = {
        'JulianDate': julian_dates,
        'Slope': slopeList,
        'ABoffset': ABoffsetList,
        'Counts': countsList,
        'LimitingMag': limiting_mags
    }
    tempDF = pd.DataFrame(data)
    # Save the DataFrame for each night
    tempDF.to_csv(f'/lustre/work/client/users/cdcook/VSPData/outputs/{field}_night{n}_summary.csv', index=False)
    print(f"DataFrame for night {n} saved successfully.")

In [ ]:
combined_data = []
for n in dates:
    file_path = f'/lustre/work/client/users/cdcook/VSPData/outputs/{field}_night{n}_summary.csv'
    print(file_path)
    tempDF = pd.read_csv(file_path)
    tempDF['Night'] = n
    combined_data.append(tempDF)

# Combine all DataFrames into one
dataDF = pd.concat(combined_data, ignore_index=True)

# Convert Julian Dates to fractional days
#data2DF['FractionalJulianDate'] = dataDF['JulianDate'] % 1
dataDF

In [ ]:
# Correct mapping of Julian Dates to nights
'''
night_mapping = {
    51823: '1006',
    51822: '1005',
    51821: '1004',
    51820: '1003',
    51819: '1002',
    51818: '1001',
    51816: '0929',
    51815: '0928',
    51793: '0906',
    51292: '0905',
    51790: '0903',
    51789: '0902',
    51788: '0901',
    51787: '0830',
    51781: '0825',
    51780: '0824'
}
'''
night_mapping = {
    51643: '409',
    51644: '410',
    51647: '413',
    51648: '414',
    51649: '415',
    51650: '416',
    51651: '417'
}
# Function to map Julian Date to correct night
def map_night(julian_date):
    date = int(julian_date)  # Extract the integer part of Julian Date
    return night_mapping.get(date, 'Unknown Night')

# Apply the mapping to create a new 'Night' column
dataDF['Night'] = dataDF['JulianDate'].apply(lambda x: map_night(x))

# Print the unique nights to verify
print("Unique nights in dataDF:", dataDF['Night'].unique())

# Convert Julian Dates to fractional days
dataDF['FractionalJulianDate'] = dataDF['JulianDate'] % 1

# Print the DataFrame to check the correct mapping
print(dataDF[['JulianDate', 'FractionalJulianDate', 'Night']].head())

In [ ]:
# Plotting with corrected night labels
plt.figure(figsize=(18, 9))
sns.scatterplot(data=dataDF, x='FractionalJulianDate', y='Slope', hue='Night', s=100)
plt.xlabel('Julian Date (Fractional Day)', fontsize=20)
plt.ylabel('Slope', fontsize=20)
plt.ylim(0.5, 1.0)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.title(f'Slope of each exposure -- {field} -- gKron 0.5', fontsize=20)
plt.legend(title='Night')
plt.show()

In [ ]:
plt.figure(figsize=(18, 9))
sns.scatterplot(data=dataDF, x='FractionalJulianDate', y='ABoffset', hue='Night', s=100)
plt.xlabel('Julian Date (Fractional Day)', fontsize=20)
plt.ylabel('AB Offset', fontsize=20)
plt.ylim(0.0, 5.0)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.title(f'AB Offset of each exposure -- {field} -- gKron 0.5', fontsize=20)
plt.legend(title='Night')
plt.show()

In [ ]:
plt.figure(figsize=(18, 9))
sns.scatterplot(data=dataDF, x='FractionalJulianDate', y='Counts', hue='Night', s=100)
plt.xlabel('Julian Date (Fractional Day)', fontsize=20)
plt.ylabel('# Matched to PanSTARRS', fontsize=20)
plt.ylim(0, 7000)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.title(f'Number of objects matched to PanSTARRS of each exposure -- {field} -- gKron 0.5', fontsize=20)
plt.legend(title='Night')
plt.show()

In [ ]:
plt.figure(figsize=(18, 9))
sns.scatterplot(data=dataDF, x='FractionalJulianDate', y='LimitingMag', hue='Night', s=100)
plt.xlabel('Julian Date (Fractional Day)', fontsize=20)
plt.ylabel('Limiting Magnitude', fontsize=20)
plt.ylim(12, 30)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.title(f'Limiting Magnitude of each exposure -- {field} -- gKron 0.5', fontsize=20)
plt.legend(title='Night')
plt.show()

In [ ]:
# Plotting multiple relationships
plt.figure(figsize=(14, 10))

# Slope vs counts
plt.subplot(2, 2, 1)
sns.scatterplot(data=dataDF, x='Slope', y='Counts', hue='Night', s=100)
plt.title(f'Slope vs Counts {field}')
plt.xlabel('Slope')
plt.ylabel('Counts')
plt.ylim(0, 7000)
plt.xlim(0.5, 1.0)

# ABoffset vs counts
plt.subplot(2, 2, 2)
sns.scatterplot(data=dataDF, x='ABoffset', y='Counts', hue='Night', s=100)
plt.title(f'ABoffset vs Counts {field}')
plt.xlabel('ABoffset')
plt.ylabel('Counts')
plt.ylim(0, 7000)
plt.xlim(0.0, 5.0)

# ABoffset vs Slope
plt.subplot(2, 2, 3)
sns.scatterplot(data=dataDF, x='ABoffset', y='Slope', hue='Night', s=100)
plt.title(f'ABoffset vs Slope {field}')
plt.xlabel('ABoffset')
plt.ylabel('Slope')
plt.ylim(0.5, 1.0)
plt.xlim(0.0, 5.0)

# Counts vs Limiting Mag
plt.subplot(2, 2, 4)
sns.scatterplot(data=dataDF, x='Counts', y='LimitingMag', hue='Night', s=100)
plt.title(f'Counts vs Limiting Magnitude {field}')
plt.xlabel('Counts')
plt.ylabel('Limiting Magnitude')
plt.ylim(12, 25)
plt.xlim(0, 7000)

plt.tight_layout()
plt.savefig(f"{field}_parplots.png")
plt.show()

In [ ]:
# Histogram plots
plt.figure(figsize=(18, 12))

# Histogram for Slope
plt.subplot(2, 2, 1)
sns.histplot(data=dataDF, x='Slope', hue='Night', multiple="stack", bins=30)
plt.title(f'Histogram of Slope {field}')
plt.xlabel('Slope')
plt.ylabel('Count')

# Histogram for ABoffset
plt.subplot(2, 2, 2)
sns.histplot(data=dataDF, x='ABoffset', hue='Night', multiple="stack", bins=30)
plt.title(f'Histogram of ABoffset {field}')
plt.xlabel('ABoffset')
plt.ylabel('Count')

# Histogram for Counts
plt.subplot(2, 2, 3)
sns.histplot(data=dataDF, x='Counts', hue='Night', multiple="stack", bins=30)
plt.title(f'Histogram of Counts {field}')
plt.xlabel('Counts')
plt.ylabel('Count')

# Histogram for Limiting Magnitude
plt.subplot(2, 2, 4)
sns.histplot(data=dataDF, x='LimitingMag', hue='Night', multiple="stack", bins=30)
plt.title(f'Histogram of Limiting Magnitude {field}')
plt.xlabel('Limiting Magnitude')
plt.ylabel('Count')

plt.tight_layout()
plt.savefig(f"{field}_histplots.png")
plt.show()